# Meme Transfer v2: entrenar el clasificador en Colab

Entrena **MobileNetV3-Large a 224 px** (el modelo "grande" del plan) con el dataset sintético de `training/synth.py` y lo exporta a ONNX para la web.

1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.
2. Ejecuta las celdas en orden.
3. Descarga `memes.onnx` y `labels.json` y súbelos a `model/` en el repo (o haz commit desde acá).

> Los Colab gratuitos tienen solo 2 CPU y el cuello de botella es **generar** las muestras, no la GPU. Por eso primero se genera un pool en disco (≈40-60 min para 120k muestras con 2 CPU) y después se entrena rápido sobre él (≈3 min por época en T4).

In [ ]:
!git clone https://github.com/santiagortegadev/memetransfer.git
%cd memetransfer
!pip -q install -r training/requirements.txt
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NINGUNA (activa la GPU)")
print("CPUs:", os.cpu_count())

## 1. Revisar el dataset sintético a ojo
Cada recorte es lo que el modelo va a ver: el interior del marco magenta después del localizador, con moiré, reflejos, perspectiva, desenfoque, etc.

In [ ]:
!cd training && python synth.py --grid /content/grid.jpg --camera-grid /content/camgrid.jpg --size 224
from IPython.display import Image, display
display(Image("/content/camgrid.jpg")); display(Image("/content/grid.jpg"))

## 2. Entrenar
`--pool` pre-genera las muestras usando todos los CPU; después se entrena en GPU. Ajusta `--pool` / `--epochs` según el tiempo disponible.

In [ ]:
!cd training && python train.py --arch large --size 224 --pool 120000 --epochs 12 --batch 128 --lr 1.5e-3 \
    --workers $(( $(nproc) - 1 )) --out runs/colab-large224

## 3. Exportar a ONNX (int8 si no pierde precisión)

In [ ]:
!cd training && python export_onnx.py runs/colab-large224/best.pt
!ls -la model/ && python -c "import json;d=json.load(open('model/labels.json'));print(d['format'], d['input_size'], d['synthetic_val'])"

## 4. (Opcional) Evaluar con un video real
Graba con el teléfono receptor un video apuntando al emisor en modo **Calibrar** (los 258 memes en orden), súbelo a Colab y ejecútalo. Criterio: ≥98 % de frames correctos entre los aceptados y <1 % de aceptaciones falsas.

In [ ]:
from google.colab import files
up = files.upload()  # elegir el video
video = next(iter(up))
!python training/eval_video.py "{video}" --export-crops /content/real_crops

Si no cumple el criterio, hay que hacer fine-tuning mezclando los recortes reales:

In [ ]:
!cd training && python train.py --arch large --size 224 --pool 120000 --epochs 4 --lr 3e-4 --batch 128 \
    --init runs/colab-large224/best.pt --real /content/real_crops --out runs/colab-large224-ft
!cd training && python export_onnx.py runs/colab-large224-ft/best.pt

## 5. Descargar el modelo
Sube `memes.onnx` y `labels.json` a la carpeta `model/` del repo. La web lee el tamaño de entrada de `labels.json`, así que no hace falta tocar código.

In [ ]:
from google.colab import files
files.download("model/memes.onnx"); files.download("model/labels.json")